In [1]:
import pandas as pd
import glob
from datetime import date, timedelta
import numpy as np
from datetime import datetime
import pathlib
from pathlib import Path
from collections import OrderedDict
import polars as pl
import fastexcel
import os
import time

In [2]:
def convert_to_datetime(struct_time):
    return datetime(*struct_time[:6])

def input_data(folder_path, sheet_name=None, filename_keyword=None):
    file_paths = glob.glob(f"{folder_path}/*.xlsx") + glob.glob(f"{folder_path}/*.csv")

    if filename_keyword:
        kw = filename_keyword.lower()
        file_paths = [f for f in file_paths if kw in os.path.basename(f).lower()]

    df_list = []
    for file in file_paths:
        export_time = os.path.getmtime(file)
        export_time_datetime = convert_to_datetime(time.localtime(export_time))

        if file.endswith('.xlsx'):
            df = pl.read_excel(file, sheet_name=sheet_name, engine='calamine')
            df = df.select(pl.all().cast(pl.String))
        elif file.endswith('.csv'):
            try:
                df = pl.read_csv(file, encoding="utf-8", infer_schema_length=0, ignore_errors=True)
            except:
                df = pl.read_csv(file, encoding="ISO-8859-1", ignore_errors=True, infer_schema_length=0)

        # Normalize column names: strip whitespace + BOM
        df.columns = [c.strip().replace('\ufeff', '').replace('\u200b', '') for c in df.columns]

        df = df.with_columns([
            pl.lit(os.path.basename(file)).alias('File Name'),
            pl.lit(export_time_datetime).alias('Export Time')
        ])
        df_list.append(df)

    if df_list:
        return pl.concat(df_list, how='diagonal_relaxed')
    return pl.DataFrame()


def input_data_parquet(folder_path):
    file_paths = glob.glob(f"{folder_path}/*.parquet")
    df_list = []
    for file in file_paths:
        export_time = os.path.getmtime(file)
        export_time_datetime = convert_to_datetime(time.localtime(export_time))
        df = pl.read_parquet(file).with_columns([
            pl.lit(os.path.basename(file)).alias('File Name'),
            pl.lit(export_time_datetime).alias('Export Time')
        ])
        df_list.append(df)
    if df_list:
        return pl.concat(df_list, how='vertical')
    return pl.DataFrame()


today_temp = datetime.today().date()
today = today_temp.strftime('%b_%d_%Y')

def pre_process_fcr_excel(folder_path: str):
    excel_files = glob.glob(os.path.join(folder_path, "*.xlsx")) + glob.glob(os.path.join(folder_path, "*.xls"))
    for file_path in excel_files:
        try:
            df = pl.read_excel(file_path, engine="calamine", has_header=False)
            if df.height < 2:
                continue
            header_row_index = 0
            for i in range(min(10, df.height)):
                row_data = [str(x).strip() for x in df.row(i) if x is not None]
                if "FCR Category" in row_data or "Conversation Id" in row_data:
                    header_row_index = i
                    break
            raw_headers = df.row(header_row_index)
            headers = []
            seen = set()
            for i, c in enumerate(raw_headers):
                col_name = str(c).strip() if c is not None and str(c).strip() != "" else f"col_{i}"
                if col_name in seen:
                    col_name = f"{col_name}_dup_{i}"
                seen.add(col_name)
                headers.append(col_name)
            df = df.slice(header_row_index + 1)
            df.columns = headers
            df.write_csv(os.path.splitext(file_path)[0] + ".csv")
        except Exception:
            pass

In [3]:
first_glob = os.path.expanduser("~").replace("\\", "/")
test_path = f"{first_glob}/Concentrix Corporation"
if not os.path.exists(test_path):
    raise FileNotFoundError(f"Not found the path: {test_path}")

folder_paths = {
    "input_performance":     f'{first_glob}/Concentrix Corporation/WFM-Expedia-HCM - Branding files/BI_Task/RAW/NEW_LOOK_EXCEL_EN/new_look_excel_data',
    "output_miv_performance":f'{first_glob}/Concentrix Corporation/WFM-Expedia-HCM - Branding files/BI_Task/MIV/MIV_Data',
    "hc_extend_by_month":    f'{first_glob}/Concentrix Corporation/WFM-Expedia-HCM - Branding files/Headcount/HC Extend by Month',
    "input_survey":          f'{first_glob}/Concentrix Corporation/WFM-Expedia-HCM - Branding files/BI_Task/RAW/SURVEY_EN',
    "input_afcr":            f'{first_glob}/Concentrix Corporation/WFM-Expedia-HCM - Branding files/BI_Task/RAW/FCR',
    "input_t3":              f'{first_glob}/Concentrix Corporation/WFM-Expedia-HCM - Branding files/BI_Task/RAW/T3_EN',
    "input_iex":             f'{first_glob}/Concentrix Corporation/WFM-Expedia-HCM - Branding files/Rawdata/STORAGE_OUTPUT_AGENT_IEX',
    "mapping_file":          f'{first_glob}/Concentrix Corporation/WFM-Expedia-HCM - Branding files/BI_Task/CODE/AQG_summarized.xlsx',
    "global_hc":             f'{first_glob}/Concentrix Corporation/WFM-Expedia-HCM - Branding files/BI_Task/CODE/Resources/Global_HC.parquet',
    "input_csv_re_direct":   f'{first_glob}/Concentrix Corporation/WFM-Expedia-HCM - Branding files/BI_Task/RAW/RE-DIRECT',
    "input_delayed_closure": f'{first_glob}/Concentrix Corporation/WFM-Expedia-HCM - Branding files/BI_Task/RAW/DELAYED_CLOSURE',
}

print("--- FULL FOLDER PATHS LIST ---")
for key, path in folder_paths.items():
    print(f"{key}: {path}")
print("-" * 60)

pre_process_fcr_excel(folder_paths["input_afcr"])

IEX = input_data(folder_paths["input_iex"]).unique()
IEX = IEX.with_columns([
    pl.col(['Date']).str.to_date("%Y-%m-%d", strict=False),
    pl.col(['Datetime_Fluctuate_Start_Shift','Datetime_Fluctuate_End_Shift',
            'Datetime_First_Start_Shift','Datetime_First_End_Shift'])
      .str.to_datetime("%Y-%m-%d %H:%M:%S.%f", strict=False),
    pl.col(['Night_Shift','Target','Unplanned','Planned',
            'Roster Presented','Roster Scheduled']).cast(pl.Float64),
])
columns_to_sec = ['Time_Of_Day','Open Time','Extra Time','Break Time',
                  'Lunch Time','Training','NCNS','AL','Target']
IEX = IEX.with_columns([
    (pl.col(col).fill_null(0).cast(pl.Float64) * 3600).alias(col) for col in columns_to_sec
])
Night_Shift_1 = IEX[['Date','Email Id','Night_Shift']].unique()
Night_Shift_2 = Night_Shift_1.with_columns(
    (pl.col('Date') - pl.duration(days=1)).alias('Previous Date')
)
Night_Shift = Night_Shift_2.join(Night_Shift_1,
    left_on=['Previous Date','Email Id'], right_on=['Date','Email Id'], how='left')
Night_Shift = Night_Shift.rename({'Night_Shift_right': 'Previous_Night_Shift'})
Previous_Date = IEX[['Date','Email Id','Datetime_First_Start_Shift','Shift Tracking']].unique()

--- FULL FOLDER PATHS LIST ---
input_performance: C:/Users/huuchinh.nguyen/Concentrix Corporation/WFM-Expedia-HCM - Branding files/BI_Task/RAW/NEW_LOOK_EXCEL_EN/new_look_excel_data
output_miv_performance: C:/Users/huuchinh.nguyen/Concentrix Corporation/WFM-Expedia-HCM - Branding files/BI_Task/MIV/MIV_Data
hc_extend_by_month: C:/Users/huuchinh.nguyen/Concentrix Corporation/WFM-Expedia-HCM - Branding files/Headcount/HC Extend by Month
input_survey: C:/Users/huuchinh.nguyen/Concentrix Corporation/WFM-Expedia-HCM - Branding files/BI_Task/RAW/SURVEY_EN
input_afcr: C:/Users/huuchinh.nguyen/Concentrix Corporation/WFM-Expedia-HCM - Branding files/BI_Task/RAW/FCR
input_t3: C:/Users/huuchinh.nguyen/Concentrix Corporation/WFM-Expedia-HCM - Branding files/BI_Task/RAW/T3_EN
input_iex: C:/Users/huuchinh.nguyen/Concentrix Corporation/WFM-Expedia-HCM - Branding files/Rawdata/STORAGE_OUTPUT_AGENT_IEX
mapping_file: C:/Users/huuchinh.nguyen/Concentrix Corporation/WFM-Expedia-HCM - Branding files/BI_Task/

C:\Users\huuchinh.nguyen\AppData\Local\Temp\ipykernel_20372\2863908424.py:32: ChronoFormatWarning: Detected the pattern `.%f` in the chrono format string. This pattern should not be used to parse values after a decimal point. Use `%.f` instead. See the full specification: https://docs.rs/chrono/latest/chrono/format/strftime
  .str.to_datetime("%Y-%m-%d %H:%M:%S.%f", strict=False),


## Survey — New Schema (Survey Dump)

Cột mapping:
- `NPS Type` → `_nps_type`, `_promoter`, `_detractor`, `_neutral`, `_survey`
- `DUET Score Type` → `DUET` (Positive=1, Negative=0)
- `Response Text EN` → `_verbatim`

Removed (no longer in new Survey Dump):
`_ir`, `_ae`, `_offer`, `d_happy_response`, `d_surprised_response`, `e/t/u_response`, `delight`, `usability`, `ease`, `trust`

In [4]:
SURVEY_INPUT = input_data(folder_paths["input_survey"], filename_keyword="Survey Dump")

SURVEY_INPUT = SURVEY_INPUT.rename({
    "Conversation_id": "Conversation Id",
    "Agent Email":     "Agent Email ID",
})

SURVEY_INPUT = SURVEY_INPUT.with_columns(
    pl.concat_str(
        [pl.col("Agent Email ID").cast(pl.Utf8), pl.col("Conversation Id").cast(pl.Utf8)],
        separator="_"
    ).alias("key_survey")
)

# NPS flags from NPS Type
survey_nps = (
    SURVEY_INPUT
    .filter(pl.col("NPS Type").is_not_null())
    .with_columns([
        pl.col("NPS Type").alias("_nps_type"),
        pl.when(pl.col("NPS Type").str.to_lowercase() == "promoter")
          .then(1).otherwise(0).alias("_promoter"),
        pl.when(pl.col("NPS Type").str.to_lowercase() == "detractor")
          .then(1).otherwise(0).alias("_detractor"),
        pl.when(pl.col("NPS Type").str.to_lowercase().is_in(["neutral", "passive"]))
          .then(1).otherwise(0).alias("_neutral"),
        pl.lit(1).alias("_survey"),
    ])
    .select(["key_survey", "Conversation Id", "_nps_type", "_promoter",
             "_detractor", "_neutral", "_survey"])
    .unique()
)

# DUET: Positive=1, Negative=0
survey_duet = (
    SURVEY_INPUT
    .filter(pl.col("DUET Score Type").is_not_null())
    .with_columns(
        pl.when(pl.col("DUET Score Type").str.to_lowercase().str.contains("positive"))
          .then(1).otherwise(0).alias("DUET")
    )
    .select(["key_survey", "Conversation Id", "DUET"])
    .unique()
)

# Verbatim
verbatim = (
    SURVEY_INPUT
    .filter(
        pl.col("Response Text EN").is_not_null() &
        (pl.col("Response Text EN").str.strip_chars() != "")
    )
    .with_columns(
        pl.col("Response Text EN")
          .str.replace_all(r"[\r\n\t]+", " ")
          .str.strip_chars()
          .alias("_verbatim")
    )
    .select(["key_survey", "Conversation Id", "_verbatim"])
    .unique()
)

# Merge all survey components
survey_final = (
    survey_nps
    .join(survey_duet, on=["key_survey", "Conversation Id"], how="left")
    .join(verbatim,    on=["key_survey", "Conversation Id"], how="left")
)

print(survey_final.shape)
survey_final.head(3)

(26849, 9)


key_survey,Conversation Id,_nps_type,_promoter,_detractor,_neutral,_survey,DUET,_verbatim
str,str,str,i32,i32,i32,i32,i32,str
"""hoangnam.trinh@concentrix.com_…","""7921b2ce-8df1-425a-9f62-2349a2…","""Detractor""",0,1,0,1,0,"""I have had multiple situations…"
"""ankit.patel5@concentrix.com_c4…","""c4148a01-be85-429a-871c-0279a5…","""Detractor""",0,1,0,1,0,"""Customer service agent was not…"
"""minhquan.le@concentrix.com_604…","""60480dea-5e13-49eb-acdb-c32ddb…","""Detractor""",0,1,0,1,0,"""nope"""


In [5]:
T3_INPUT = input_data(folder_paths["input_t3"], filename_keyword="T3_CNX_AWS")

t3_final = (
    T3_INPUT
    .with_columns(
        pl.when(
            pl.col("Transfer Destination").str.contains("Tier 3", literal=True)
        ).then(1).otherwise(0).alias("T3")
    )
    .filter(pl.col("T3") == 1)
    .with_columns(
        pl.concat_str(
            [pl.col("Agent Email ID").cast(pl.Utf8), pl.col("Conversation Id").cast(pl.Utf8)],
            separator="_"
        ).alias("key_t3")
    )
    .select(["key_t3", "T3"])
    .unique()
)
t3_final.head(3)

key_t3,T3
str,i32
"""nguyendananh.le@concentrix.com…",1
"""abhishek.mazumdar@concentrix.c…",1
"""shatadru.chowdhury@concentrix.…",1


In [6]:
DELAYED_CLOSURE_INPUT = (
    input_data(folder_paths["input_delayed_closure"], filename_keyword="excess_aws")
    .unique(subset=["User Email", "Conversation ID"], keep="last")
)

delayed_closure = (
    DELAYED_CLOSURE_INPUT
    .select([
        "User Email", "Conversation ID",
        "Excess Time", "Disconnected Reason (groups)",
        "last_traveler_message_sent_datetime_utc",
    ])
    .rename({
        "User Email":      "Agent Email ID",
        "Conversation ID": "Conversation Id",
        "Excess Time":     "_excess_time_raw",
    })
    .with_columns(
        pl.col("_excess_time_raw").cast(pl.Float64).fill_null(0).alias("Exceed Time"),
    )
    .with_columns([
        (pl.col("Exceed Time") > 0).cast(pl.Int8).alias("Exceed Chat"),
        pl.col("Disconnected Reason (groups)").str.to_lowercase()
          .str.contains("agent").cast(pl.Int8).alias("Agent Disconnect"),
        pl.col("Disconnected Reason (groups)").str.to_lowercase()
          .str.contains("ghost").cast(pl.Int8).alias("Ghost"),
        pl.col("Disconnected Reason (groups)").str.to_lowercase()
          .str.contains("requeue").cast(pl.Int8).alias("Requeued"),
        pl.col("last_traveler_message_sent_datetime_utc")
          .is_null().cast(pl.Int8).alias("Traveler Unresponsive"),
        pl.concat_str(
            [pl.col("Agent Email ID").cast(pl.Utf8), pl.col("Conversation Id").cast(pl.Utf8)],
            separator="_"
        ).alias("key_delayed_closure"),
    ])
    .drop(["_excess_time_raw", "Disconnected Reason (groups)",
           "last_traveler_message_sent_datetime_utc"])
    .group_by(["Agent Email ID", "Conversation Id", "key_delayed_closure"])
    .agg([
        pl.sum("Exceed Time"),
        pl.sum("Exceed Chat"),
        pl.sum("Agent Disconnect"),
        pl.sum("Ghost"),
        pl.sum("Requeued"),
        pl.sum("Traveler Unresponsive"),
    ])
)

_em = pl.col("Exceed Time") / 60
delayed_closure = delayed_closure.with_columns(
    pl.when(pl.col("Exceed Time") <= 0).then(pl.lit(None, dtype=pl.Utf8))
      .when(_em <= 1).then(pl.lit("01 Mins"))
      .when(_em <= 2).then(pl.lit("02 Mins"))
      .when(_em <= 3).then(pl.lit("03 Mins"))
      .when(_em <= 4).then(pl.lit("04 Mins"))
      .when(_em <= 5).then(pl.lit("05 Mins"))
      .when(_em <= 10).then(pl.lit("05-10 Mins"))
      .when(_em <= 15).then(pl.lit("10-15 Mins"))
      .when(_em <= 30).then(pl.lit("15-30 Mins"))
      .otherwise(pl.lit("30+ Min"))
      .alias("Exceed Bucket")
)
print(delayed_closure.height)

37143


In [7]:
def process_afcr_folder(folder_path: str) -> pl.DataFrame:
    all_dataframes = []
    for file_path in glob.glob(os.path.join(folder_path, "*.csv")):
        file_name   = os.path.basename(file_path)
        export_time = datetime.fromtimestamp(os.path.getmtime(file_path)).strftime('%Y-%m-%d %H:%M:%S')
        df = pl.read_csv(
            file_path,
            encoding="utf-8",
            schema_overrides={"Itinerary": pl.String},
            infer_schema_length=10000,
            ignore_errors=True
        )
        cast_exprs = []
        for col, dtype in [("Handle Time", pl.Float64), ("Duet", pl.Float64),
                           ("Passed Sessions", pl.Int64), ("Failed Sessions", pl.Int64)]:
            if col in df.columns:
                cast_exprs.append(pl.col(col).cast(dtype, strict=False))
        if cast_exprs:
            df = df.with_columns(cast_exprs)
        df = df.with_columns([
            pl.lit(file_name).alias('File Name'),
            pl.lit(export_time).alias('Export Time')
        ])
        all_dataframes.append(df)
    if not all_dataframes:
        return pl.DataFrame()
    return pl.concat(all_dataframes, how="diagonal_relaxed")


afcr_input = process_afcr_folder(folder_paths["input_afcr"])

if not afcr_input.is_empty():
    afcr_input = (
        afcr_input
        .filter(pl.col("Vendor Partner Location") == "Concentrix (Ho Chi Minh City)")
        .select([
            pl.col("Agent Email Address").alias("Agent Email ID"),
            pl.col("Conversation Id"),
            pl.col("Passed Sessions"),
            pl.col("Failed Sessions"),
            pl.when(pl.col("Passed Sessions").is_in([0, 1])).then(1).otherwise(0).alias("Total Sessions")
        ])
        .unique()
    )

In [8]:
RE_DIRECT_INPUT = input_data(folder_paths["input_csv_re_direct"])

re_direct_final = (
    RE_DIRECT_INPUT
    .with_columns(
        Re_Direct=pl.lit(1),
        **{
            "Re-Direct Text": (
                pl.col("Text").cast(pl.Utf8).fill_null("")
                  .str.replace_all(r"(\r\n|\r|\n)+", " | ")
                  .str.replace_all(r"[\-•\u2022\u25CF\u25E6\u2043\u2219\u00B7\u2013\u2014]+", "")
                  .str.replace_all(r"\s{2,}", " ")
                  .str.strip_chars()
            )
        }
    )
    .with_columns(
        pl.concat_str(
            [pl.col("Agent People Id").cast(pl.Utf8), pl.col("Conversation Id").cast(pl.Utf8)],
            separator="_"
        ).alias("key_redirect")
    )
    .select(["key_redirect", "Agent People Id", "Re_Direct", "Re-Direct Text"])
    .unique(subset=["key_redirect"], maintain_order=True)
)

In [9]:
PERFORMANCE_INPUT = input_data(
    folder_paths["input_performance"],
    filename_keyword="aws_performance_retail_rawdata"
)

try:
    if PERFORMANCE_INPUT.columns[0] == "":
        PERFORMANCE_INPUT = PERFORMANCE_INPUT.drop(PERFORMANCE_INPUT.columns[0])
except: pass

print(PERFORMANCE_INPUT.columns)

# ── Joined Time parser ─────────────────────────────────────────────────────
def _build_joined_time(time_col: str = "Connected To Agent Time") -> pl.Expr:
    raw = pl.col(time_col).cast(pl.Utf8).str.strip_chars()
    return (
        pl.coalesce([
            raw.str.strptime(pl.Datetime, "%Y-%m-%d %H:%M:%S", strict=False),
            raw.str.strptime(pl.Datetime, "%m/%d/%Y %H:%M",    strict=False),
        ])
        .alias("Joined Time")
    )

# ── Duration parser (seconds or HH:MM:SS) ─────────────────────────────────
def _duration_to_seconds(col: str) -> pl.Expr:
    raw       = pl.col(col).cast(pl.Utf8).str.strip_chars()
    as_number = raw.str.replace_all(",", "").cast(pl.Float64, strict=False)
    h = raw.str.extract(r"^(\d+):\d{2}:\d{2}$", 1).cast(pl.Float64, strict=False)
    m = raw.str.extract(r"^\d+:(\d{2}):\d{2}$", 1).cast(pl.Float64, strict=False)
    s = raw.str.extract(r"^\d+:\d{2}:(\d{2})$", 1).cast(pl.Float64, strict=False)
    from_hhmmss = h * 3600 + m * 60 + s
    return pl.when(as_number.is_not_null()).then(as_number).otherwise(from_hhmmss).alias(col)

# ── AWS column adapter ─────────────────────────────────────────────────────
PERFORMANCE_INPUT = (
    PERFORMANCE_INPUT
    .with_columns(_build_joined_time())
    .rename({
        "Handle Time":                    "Handle Time (Sum)",
        "Talk Time":                      "Talk Time (Sum)",
        "Acw Duration":                   "Wrap Up Time (Sum)",
        "Agent Vendor Location":          "Agent Business Location",
        "Outbound Initiated (Yes / No)":  "Initiated Outbound (Yes / No)",
        "Product":                        "Latest VA Product",
        "Intent":                         "Latest VA Intent",
        "Locale":                         "Language",
    })
    .with_columns([
        pl.col("Latest VA Product").fill_null("UNKNOWN").alias("Latest VA Product"),
        pl.col("Latest VA Intent").fill_null("UNKNOWN").alias("Latest VA Intent"),
    ])
)

print(PERFORMANCE_INPUT.select(["Connected To Agent Time", "Joined Time"]).head(5))

existing_cols = set(PERFORMANCE_INPUT.columns)

columns_to_cast = {
    "Handle Time (Sum)":  pl.Float64,
    "Talk Time (Sum)":    pl.Float64,
    "Wrap Up Time (Sum)": pl.Float64,
    "Hold Time (Sum)":    pl.Float64,
    "Handle (Count)":     pl.Int64,
}

casts = []
for _col, _dtype in columns_to_cast.items():
    if _col not in existing_cols:
        continue
    if _dtype == pl.Float64:
        casts.append(_duration_to_seconds(_col))
    else:
        casts.append(pl.col(_col).cast(_dtype, strict=False).alias(_col))

PERFORMANCE_CHANGED_TYPE = PERFORMANCE_INPUT.with_columns(casts)

# ── Routing Profile → LOB + Agent Queue Group Name override ───────────────
LG_CHAT_PROFILES = [
    "Chat_AC_GLB_EN_Car_Activity",
    "Chat_AC_GLB_EN_Lodging_Nesting",
    "Chat_AC_GLB_EN_Lodging_Proficient",
]
NL_CHAT_PROFILES = [
    "Chat_AC_GLB_EN_Proficient",
    "Chat_AC_GLB_EN_NL_Nesting",
]

PERFORMANCE_CHANGED_TYPE = PERFORMANCE_CHANGED_TYPE.with_columns([
    pl.col("Agent Routing Profile Name").alias("Agent Queue Group Name"),
    pl.when(pl.col("Agent Routing Profile Name").is_in(LG_CHAT_PROFILES))
      .then(pl.lit("LG Chat"))
      .when(pl.col("Agent Routing Profile Name").is_in(NL_CHAT_PROFILES))
      .then(pl.lit("NL Chat"))
      .otherwise(pl.col("Agent Routing Profile Name"))
      .alias("LOB"),
])

PERFORMANCE_NEXT_STEP = PERFORMANCE_CHANGED_TYPE.with_columns([
    pl.col("Joined Time").dt.date().alias("Joined Date")
])
PERFORMANCE_NEXT_STEP = PERFORMANCE_NEXT_STEP.with_columns([
    (pl.col("Joined Time") + pl.duration(hours=14)).alias("Join Time (VNT)"),
    (pl.col("Joined Time") + pl.duration(hours=14)).dt.date().alias("Join Date (VNT)")
])

['Connected To Agent Date', 'Agent People Id', 'Conversation Id', 'Agent Email ID', 'Agent Routing Profile Name', 'Agent Vendor Location', 'Outbound Initiated (Yes / No)', 'Business Segment Name', 'Partner Name', 'Locale', 'Connected To Agent Time', 'Agent Name', 'Agent Queue Group Name', 'Agent Manager Name', 'Handle Time', 'Talk Time', 'Assigned Agent Time', 'Acw Duration', 'Offered', 'Requeued (Yes / No)', 'Response Time', 'Response Count', 'Actual Disconnect Reason', 'Requeue Time', 'Inbound Message Count', 'Outbound Message Count', 'Transfer Initiated (Yes / No)', 'Agent Tenure in Days', 'Initial Channel Type', 'Answer Time', 'Queue Time', 'Intent', 'Product', 'Contact Disconnect (Count)', 'Handle (Count)', 'Hold Time (Sum)', 'File Name', 'Export Time']
shape: (5, 2)
┌─────────────────────────┬─────────────────────┐
│ Connected To Agent Time ┆ Joined Time         │
│ ---                     ┆ ---                 │
│ str                     ┆ datetime[μs]        │
╞════════════════

In [10]:
HC_MASTER_DATABASE = input_data(folder_paths["hc_extend_by_month"])
HC_MASTER_DATABASE = HC_MASTER_DATABASE.rename({'Date Start Week': 'Week_Monday'})
HC_MASTER_DATABASE = HC_MASTER_DATABASE.with_columns([
    pl.col('Date').str.strptime(pl.Date, "%Y-%m-%d", strict=False)
])

hc_master_selected = HC_MASTER_DATABASE.select([
    "Date","Email Id","OracleID","People ID","IEX ID","Employee Name","Alias","Designation",
    "Detail Status","Active","TL ID","Supervisor Email",
    "Supervisor Name","Wave","LOB",'LG Tenure','NL Tenure',
    'Mini TL - Email','Mini TL - Short Name','Mini TL Start Date','Site'
]).unique()

hc_master_selected = hc_master_selected.rename({'Mini TL - Short Name': 'Mini TL', 'LOB': 'Group'})

performance_merged = PERFORMANCE_NEXT_STEP.join(
    hc_master_selected,
    left_on=["Joined Date","Agent Email ID"],
    right_on=["Date","Email Id"],
    how="left"
)

GLOBAL_HC = pl.read_parquet(folder_paths["global_hc"])
global_hc_clean = GLOBAL_HC.select(["SSO ID","Production Start date","Agent/Non Agent"]).unique(subset=["SSO ID"], keep="first")
merged_global_hc = performance_merged.join(global_hc_clean, left_on="Agent Email ID", right_on="SSO ID", how="left")

merged_iex = merged_global_hc.join(
    IEX[['Date','Email Id','First Shift','Datetime_First_Start_Shift','Night_Shift']],
    left_on=['Join Date (VNT)','Agent Email ID'],
    right_on=['Date','Email Id'],
    how='left'
)

mapping   = pl.read_excel(folder_paths["mapping_file"])
lc_mapping = pl.read_excel(folder_paths["mapping_file"], sheet_name="kpi")
performance_cleaned = merged_iex.join(mapping, on="Agent Queue Group Name", how='left')

Could not determine dtype for column 5, falling back to string


In [11]:
performance_updated_ns = performance_cleaned.with_columns(
    (pl.col('Join Date (VNT)') - pl.duration(days=1)).alias('Previous Date')
)
performance_updated_ns = performance_updated_ns.join(
    Night_Shift[['Date','Email Id','Night_Shift','Previous_Night_Shift']],
    left_on=['Join Date (VNT)','Agent Email ID'],
    right_on=['Date','Email Id'],
    how='left'
)

def update_night_shift(df: pl.DataFrame) -> pl.DataFrame:
    df = df.with_columns(
        pl.when(
            (pl.col('Night_Shift') == 0) &
            (pl.col('Join Time (VNT)').dt.time() < pl.time(17, 0)) &
            (pl.col('Join Time (VNT)').dt.time() >= pl.time(0, 0)) &
            (pl.col('Previous_Night_Shift') == 1)
        ).then(False).otherwise(True).alias('Night_Shift_2_Check')
    )
    df = df.with_columns(
        pl.when(pl.col('Night_Shift_2_Check') == False)
          .then(1).otherwise(pl.col('Night_Shift')).alias('Night_Shift')
    )
    df = df.with_columns(
        pl.when((pl.col('Join Time (VNT)').dt.hour() >= 0) & (pl.col('Join Time (VNT)').dt.hour() < 12) & (pl.col('Previous_Night_Shift') == 1))
          .then(pl.col('Join Date (VNT)') - pl.duration(days=1))
        .when((pl.col('Join Time (VNT)').dt.hour() >= 0) & (pl.col('Join Time (VNT)').dt.hour() < 12) & (pl.col('Night_Shift') == 1))
          .then(pl.col('Join Date (VNT)') - pl.duration(days=1))
        .when((pl.col('Join Time (VNT)').dt.hour() >= 0) & (pl.col('Join Time (VNT)').dt.hour() < 18) & (pl.col('Night_Shift') == 0))
          .then(pl.col('Join Date (VNT)'))
        .when((pl.col('Join Time (VNT)').dt.hour() >= 18) & (pl.col('Night_Shift') == 1))
          .then(pl.col('Join Date (VNT)'))
        .otherwise(pl.col('Join Date (VNT)'))
        .alias('_Date_Converted')
    )
    return df

performance_updated_ns = update_night_shift(performance_updated_ns)
performance_updated_ns.select(["Joined Time","Join Time (VNT)","Join Date (VNT)","_Date_Converted"]).head(5)

Joined Time,Join Time (VNT),Join Date (VNT),_Date_Converted
datetime[μs],datetime[μs],date,date
2026-06-30 09:07:53,2026-06-30 23:07:53,2026-06-30,2026-06-30
2026-06-30 00:46:17,2026-06-30 14:46:17,2026-06-30,2026-06-30
2026-06-30 06:07:23,2026-06-30 20:07:23,2026-06-30,2026-06-30
2026-06-30 20:30:30,2026-07-01 10:30:30,2026-07-01,2026-07-01
2026-06-30 14:02:06,2026-07-01 04:02:06,2026-07-01,2026-07-01


In [12]:
# NOTE: _promoter/_detractor/_neutral/_survey/_nps_type now come from survey_final join.
# CCR72/_fup_72/_rr/_offer/_ir/_ae removed — no source columns in new schema.
performance_processed = performance_updated_ns.with_columns([
    pl.when(pl.col("Initiated Outbound (Yes / No)") == "Yes").then(1).otherwise(0).alias("_aob"),

    pl.col("Joined Time").dt.date().alias("_PST.Date"),
    pl.col("Joined Time").dt.strftime("%y_%m").alias("_PST.Month"),
    pl.col("Joined Time").dt.week().alias("_PST.Week"),
    pl.col("Joined Time").dt.year().alias("_PST.Year"),

    pl.concat_str([
        pl.col("Agent Email ID").cast(pl.Utf8).fill_null(""),
        pl.col("Conversation Id").cast(pl.Utf8).fill_null(""),
        pl.col("Joined Time").dt.strftime("%y%m%d%H%M%S")
    ], separator="_").alias("_conver_unique"),

    pl.when(pl.col("Group").is_in(["Non_Lodging", "Lodging"]))
      .then(pl.lit("agent")).otherwise(None).alias("Agent"),
    pl.col("_Date_Converted").alias("_Date"),
    pl.when(pl.col("Group") == "Lodging").then(pl.lit("LG Tenure"))
      .when(pl.col("Group") == "Non_Lodging").then(pl.lit("NL Tenure"))
      .otherwise(None).alias("Tenure"),
])

# AON Days + Status
performance_processed = performance_processed.with_columns(
    (
        pl.col("_PST.Date").cast(pl.Date) -
        pl.col("Production Start date").cast(pl.String)
          .str.to_date("%Y-%m-%d %H:%M:%S", strict=False).cast(pl.Date)
    ).dt.total_days().cast(pl.Int32).alias("AON_Days")
)
performance_processed = performance_processed.with_columns([
    pl.when(
        pl.col("AON_Days").is_null() &
        (pl.col("Agent/Non Agent").is_in(["Agent","ID Deleted"]))
    ).then(pl.lit("Nesting"))
      .when(pl.col("AON_Days") > 180).then(pl.lit("> 180 Days"))
      .when(pl.col("AON_Days") >= 91).then(pl.lit("91 - 180"))
      .when(pl.col("AON_Days") >= 61).then(pl.lit("61 - 90"))
      .when(pl.col("AON_Days") >= 31).then(pl.lit("31 - 60"))
      .when(pl.col("AON_Days") >= 0).then(pl.lit("00 - 30"))
      .otherwise(None).alias("AON Status")
])

# LC threshold
performance_processed = performance_processed.join_asof(
    lc_mapping, left_on="_PST.Date", right_on="Effective Date", by="LOB", strategy="backward"
)
performance_processed = performance_processed.with_columns([
    (pl.col("Handle Time (Sum)") >= pl.col("Threshole_LC")).cast(pl.Int8).alias("_lc"),
    (pl.col("Handle Time (Sum)") < 240).cast(pl.Int8).alias("Short Chat"),
])

# Composite keys
performance_processed = performance_processed.with_columns([
    pl.concat_str([pl.col("Agent Email ID"), pl.col("_PST.Date").dt.strftime("%y%m%d")]).alias("KEY"),
    pl.concat_str([pl.col("Agent Email ID").cast(pl.Utf8), pl.col("Conversation Id").cast(pl.Utf8)], separator="_").alias("EmailID_ConversationID_KEY"),
    pl.concat_str([pl.col("OracleID").cast(pl.Utf8), pl.col("Conversation Id").cast(pl.Utf8)], separator="_").alias("OracleID_ConversationID_KEY"),
    pl.concat_str([pl.col("Agent People Id").cast(pl.Utf8), pl.col("Conversation Id").cast(pl.Utf8)], separator="_").alias("PeopleID_ConversationID_KEY"),
])

# 30-min interval + period (native Polars, no map_elements)
_pst_hour = pl.col("Joined Time").dt.hour()
_pst_min  = pl.col("Joined Time").dt.minute()
_vnt_hour = pl.col("Join Time (VNT)").dt.hour()
_vnt_min  = pl.col("Join Time (VNT)").dt.minute()
_shift_h  = pl.col("Datetime_First_Start_Shift").dt.hour()

def _interval(h, m):
    sm = pl.when(m < 30).then(pl.lit(0)).otherwise(pl.lit(30))
    em = pl.when(m < 30).then(pl.lit(29)).otherwise(pl.lit(59))
    return pl.concat_str([
        h.cast(pl.Utf8).str.zfill(2), pl.lit(":"),
        sm.cast(pl.Utf8).str.zfill(2), pl.lit("-"),
        h.cast(pl.Utf8).str.zfill(2), pl.lit(":"),
        em.cast(pl.Utf8).str.zfill(2),
    ])

performance_processed = performance_processed.with_columns([
    _interval(_pst_hour, _pst_min).alias("_PST.Interval"),
    _interval(_vnt_hour, _vnt_min).alias("_VNT.Interval"),
    pl.when(_shift_h >= 18).then(pl.lit("Night"))
      .when(_shift_h >= 12).then(pl.lit("Mid"))
      .otherwise(pl.lit("Morning")).alias("_VNT.Period"),
])

C:\Users\huuchinh.nguyen\AppData\Local\Temp\ipykernel_20372\4166081039.py:47: UserWarning: Sortedness of columns cannot be checked when 'by' groups provided
  performance_processed = performance_processed.join_asof(


In [13]:
performance_merged_survey_t3 = (
    performance_processed
    .join(t3_final,                                    left_on="_conver_unique",              right_on="key_t3",               how="left")
    .join(delayed_closure,                             left_on="EmailID_ConversationID_KEY",  right_on="key_delayed_closure",  how="left")
    .join(re_direct_final,                             left_on="PeopleID_ConversationID_KEY", right_on="key_redirect",         how="left")
    .join(survey_final.drop("Conversation Id"),        left_on="EmailID_ConversationID_KEY",  right_on="key_survey",           how="left")
    .join(afcr_input,                                  on=["Agent Email ID", "Conversation Id"],                               how="left")
)

print(performance_merged_survey_t3.columns)
print(performance_merged_survey_t3.select(["_PST.Date","Production Start date","AON_Days"]).head())

selected_columns = [
    "Export Time","File Name","Agent People Id","Business Segment Name","Partner Name",
    "Response Count","Response Time","Latest VA Product","Language","Latest VA Intent","Conversation Id",
    "Agent Queue Group Name","Joined Time","_PST.Interval","Agent Email ID",
    "Handle (Count)","Handle Time (Sum)","Hold Time (Sum)","Talk Time (Sum)",
    "Join Time (VNT)","_VNT.Interval","_VNT.Period","Wrap Up Time (Sum)","Agent Business Location",
    "_PST.Date","_PST.Month","_PST.Year","_aob","LOB","_conver_unique",
    "_nps_type","_promoter","_detractor","_neutral","_survey",
    "DUET","_verbatim",
    "T3","Re_Direct","Re-Direct Text",
    "Exceed Time","Exceed Chat","Exceed Bucket",
    "Agent Disconnect","Ghost","Requeued","Traveler Unresponsive",
    "_PST.Week","_lc","AON Status","Agent/Non Agent","Tenure",
    "OracleID","People ID","IEX ID","Employee Name","Alias","Designation",
    "Detail Status","Active","TL ID","Supervisor Email","Supervisor Name",
    "Wave","Group","_Date","Mini TL - Email","Mini TL","Mini TL Start Date","Site",
    "Short Chat","Passed Sessions","Failed Sessions","Total Sessions",
]

fcr_columns = [
    "LOB","OracleID_ConversationID_KEY","EmailID_ConversationID_KEY",
    "_nps_type","_detractor","_survey","_PST.Week","AON Status",
    "Agent/Non Agent","Tenure","Employee Name","Alias","Designation","Detail Status",
    "TL ID","Supervisor Name","Supervisor Email","Wave","Group","_Date",
    "Mini TL - Email","Mini TL","Mini TL Start Date","Site","Agent Business Location",
]

missing_cols = [col for col in selected_columns if col not in performance_merged_survey_t3.columns]
print("Missing columns:", missing_cols)

performance_filtered = performance_merged_survey_t3.select(selected_columns).unique()

performance_filtered = performance_filtered.sort(
    by=["Conversation Id","Agent Email ID","Joined Time"],
    descending=[False, False, True]
).with_columns(
    (pl.col("Joined Time").cum_count().over(["Conversation Id","Agent Email ID"]) > 1)
    .cast(pl.Int8).alias("Duplicate_Flag")
)

['Connected To Agent Date', 'Agent People Id', 'Conversation Id', 'Agent Email ID', 'Agent Routing Profile Name', 'Agent Business Location', 'Initiated Outbound (Yes / No)', 'Business Segment Name', 'Partner Name', 'Language', 'Connected To Agent Time', 'Agent Name', 'Agent Queue Group Name', 'Agent Manager Name', 'Handle Time (Sum)', 'Talk Time (Sum)', 'Assigned Agent Time', 'Wrap Up Time (Sum)', 'Offered', 'Requeued (Yes / No)', 'Response Time', 'Response Count', 'Actual Disconnect Reason', 'Requeue Time', 'Inbound Message Count', 'Outbound Message Count', 'Transfer Initiated (Yes / No)', 'Agent Tenure in Days', 'Initial Channel Type', 'Answer Time', 'Queue Time', 'Latest VA Intent', 'Latest VA Product', 'Contact Disconnect (Count)', 'Handle (Count)', 'Hold Time (Sum)', 'File Name', 'Export Time', 'Joined Time', 'LOB', 'Joined Date', 'Join Time (VNT)', 'Join Date (VNT)', 'OracleID', 'People ID', 'IEX ID', 'Employee Name', 'Alias', 'Designation', 'Detail Status', 'Active', 'TL ID', 'S

In [14]:
performance_filtered = performance_filtered.sort(
    by=["Conversation Id","Agent Email ID","Joined Time"],
    descending=[False, False, True]
).with_columns(
    (pl.col("Joined Time").cum_count().over(["Conversation Id","Agent Email ID"]) > 1)
    .cast(pl.Int8).alias("IDs Removed")
)

In [15]:
# -------------------------------------------------------------------------------------
# DUET dedup: if a Conversation Id has >3 rows with DUET=0,
# keep only the 1st as 0 and nullify the rest.
# Logic unchanged — DUET is now binary (0/1) from DUET Score Type.
# -------------------------------------------------------------------------------------
performance_filtered = performance_filtered.sort(["Conversation Id"])

performance_filtered = performance_filtered.with_columns([
    (pl.col("DUET") == 0).alias("_is_zero")
])

performance_filtered = performance_filtered.with_columns([
    pl.col("_is_zero").sum().over("Conversation Id").alias("_zero_total"),
    pl.col("_is_zero").cum_sum().over("Conversation Id").alias("_zero_rank"),
])

performance_filtered = performance_filtered.with_columns(
    pl.when(
        (pl.col("_is_zero")) &
        (pl.col("_zero_total") > 3) &
        (pl.col("_zero_rank") > 1)
    ).then(None)
     .otherwise(pl.col("DUET"))
     .alias("DUET")
).drop(["_is_zero","_zero_total","_zero_rank"])

In [16]:
performance_unique = performance_filtered.drop(["Export Time","File Name"]).unique()

performance_hcm      = performance_unique.filter(pl.col("Agent Business Location").str.contains("Ho Chi Minh", literal=True))
performance_all_site = performance_unique

# Export per month + total parquet (diagonal_relaxed để merge với tháng cũ)
output_dir   = folder_paths["output_miv_performance"]
os.makedirs(output_dir, exist_ok=True)

for (month_value,), group in performance_hcm.group_by(['_PST.Month'], maintain_order=True):
    base_name = str(month_value)
    group.write_csv(os.path.join(output_dir, f"{base_name}.csv"))

parquet_file     = "_miv_performance_hcm.parquet"
out_path_parquet = os.path.join(output_dir, parquet_file)
performance_hcm.write_parquet(out_path_parquet)

print(f"Exported {performance_hcm.height:,} rows → {out_path_parquet}")

Exported 81,988 rows → C:/Users/huuchinh.nguyen/Concentrix Corporation/WFM-Expedia-HCM - Branding files/BI_Task/MIV/MIV_Data\_miv_performance_hcm.parquet


In [17]:
print(performance_all_site['_PST.Month'].drop_nulls().unique().sort())
print(performance_all_site.schema)

shape: (2,)
Series: '_PST.Month' [str]
[
	"26_06"
	"26_07"
]
Schema({'Agent People Id': String, 'Business Segment Name': String, 'Partner Name': String, 'Response Count': String, 'Response Time': String, 'Latest VA Product': String, 'Language': String, 'Latest VA Intent': String, 'Conversation Id': String, 'Agent Queue Group Name': String, 'Joined Time': Datetime(time_unit='us', time_zone=None), '_PST.Interval': String, 'Agent Email ID': String, 'Handle (Count)': Int64, 'Handle Time (Sum)': Float64, 'Hold Time (Sum)': Float64, 'Talk Time (Sum)': Float64, 'Join Time (VNT)': Datetime(time_unit='us', time_zone=None), '_VNT.Interval': String, '_VNT.Period': String, 'Wrap Up Time (Sum)': Float64, 'Agent Business Location': String, '_PST.Date': Date, '_PST.Month': String, '_PST.Year': Int32, '_aob': Int32, 'LOB': String, '_conver_unique': String, '_nps_type': String, '_promoter': Int32, '_detractor': Int32, '_neutral': Int32, '_survey': Int32, 'DUET': Int32, '_verbatim': String, 'T3': Int32,